# Практическая работа №2. Часть 2Н
## Инициализация весов и нормализация

**Цель:** увидеть на опыте, почему глубокая сеть с неудачными начальными весами не учится, написать свои функции инициализации Ксавье и Кайминга, отсечение градиента и собственный слой пакетной нормализации с двумя режимами работы, сверить их с PyTorch и проверить действие нормализации на настоящей сверточной сети.

> **Правило части.** В заданиях Н1–Н4 инициализация и отсечение градиента пишутся на тензорах, без `torch.nn.init` и `torch.nn.utils` (готовые функции используются только для сверки). В Н5 слой пакетной нормализации пишется как наследник `nn.Module`, готовые `nn.BatchNorm1d` и `nn.BatchNorm2d` нужны только для сверки. В Н6 используется готовый `nn.BatchNorm2d`.

**Задача:** сквозные примеры со слайдов. Первый: полносвязный слой с 256 входами, через который проходит сигнал с дисперсией 1; от масштаба весов зависит, затухнет сигнал, сохранится или взорвется. Второй: выход одного нейрона на пакете из четырех объектов $(3, 5, 5, 11)$, который нормализуется вручную. В конце сеть `SimpleCNN` из части 4А обучается на FashionMNIST с пакетной нормализацией и без нее.

**Как работать с ноутбуком.** В ячейках с заданиями замените `...` своим кодом, подсказка написана в комментарии `# <--`. Напечатанное значение должно совпасть с тем, что стоит после слова «ожидается». Там, где результат зависит от случайных чисел, указано примерное значение или качественное ожидание. Сначала прочитайте теорию ниже.

## Как устроены инициализация и нормализация

**1. Симметрия: почему нельзя начинать с одинаковых весов.** Если два нейрона скрытого слоя получают одни и те же входы и начинают с одинаковых весов, они вычисляют одно и то же. Значит, обратное распространение даст им одинаковые градиенты, после шага их веса снова совпадут, и так будет на каждом шаге (Гудфеллоу, раздел 8.4). Слой из сотни нейронов будет работать как один нейрон.

*Пример.* Сеть $2 \to 2 \to 1$ с функцией активации ReLU, без смещений, все веса равны $0{,}5$. Вход $x = (1, 2)$, правильный ответ $y = 1$, потеря $L = (\hat y - y)^2$.
- Прямой проход: у обоих скрытых нейронов $z = 0{,}5 \cdot 1 + 0{,}5 \cdot 2 = 1{,}5$, $h = \mathrm{ReLU}(1{,}5) = 1{,}5$; выход $\hat y = 0{,}5 \cdot 1{,}5 + 0{,}5 \cdot 1{,}5 = 1{,}5$.
- Обратный проход: $\partial L / \partial \hat y = 2(1{,}5 - 1) = 1$. Градиент весов выходного нейрона $(h_1, h_2) = (1{,}5;\ 1{,}5)$. Градиент строки весов каждого скрытого нейрона $1 \cdot 0{,}5 \cdot 1 \cdot x = (0{,}5;\ 1)$.
- После шага с $\eta = 0{,}1$ обе строки первого слоя равны $(0{,}45;\ 0{,}4)$: нейроны остались копиями.

Нулевые веса еще хуже: при $W = 0$ все скрытые выходы равны нулю, и градиенты всех весов, кроме смещения выхода, тоже нулевые, сеть вообще не сдвигается с места. Поэтому веса начинают со **случайных** значений, это и называют нарушением симметрии. Смещения обычно начинают с нуля: симметрию уже нарушили веса.

**2. Как сигнал проходит через слой.** Выход нейрона $z = \sum_{i=1}^{n} w_i x_i$. Пусть веса и входы независимы, их средние равны нулю, дисперсия весов $\sigma^2$, дисперсия входов $v$. Дисперсия произведения $w_i x_i$ равна $\sigma^2 v$, а дисперсия суммы независимых слагаемых равна сумме дисперсий, поэтому

$$\mathrm{Var}(z) = n\,\sigma^2\,v.$$

За один слой дисперсия умножается на $n\sigma^2$, а стандартное отклонение на $\sqrt{n}\,\sigma$. В сети из $L$ таких слоев без функции активации множители перемножаются:

$$\mathrm{std}_L = \left(\sqrt{n}\,\sigma\right)^L \cdot \mathrm{std}_0.$$

Пример со слайдов, $n = 256$, $\sqrt{n} = 16$:
- $\sigma = 0{,}05$: множитель $16 \cdot 0{,}05 = 0{,}8$, после 20 слоев $0{,}8^{20} \approx 0{,}0115$, сигнал **затух**;
- $\sigma = 0{,}1$: множитель $1{,}6$, после 20 слоев $1{,}6^{20} \approx 12\,089$, сигнал **взорвался**;
- $\sigma = 1/16 = 0{,}0625$: множитель ровно 1, сигнал **сохраняется** на любой глубине.

Функция активации меняет картину. Около нуля $\tanh z \approx z$, поэтому слишком маленький сигнал в сети с tanh затухает так же, как в линейной. При большом $|z|$ значения tanh прилипают к $\pm 1$, это **насыщение**: производная $1 - \tanh^2 z$ почти равна нулю, и градиент через такой нейрон не проходит. Та же история при обратном проходе: градиент по входу слоя $\delta_x = W^\top \delta_z$, и его дисперсия умножается на $n_{out}\,\sigma^2$, где $n_{out}$ это число выходов слоя. Поэтому затухание или взрыв сигнала вперед обычно сопровождается затуханием или взрывом градиента назад (разделы 8.2.5 и 8.4).

**3. Инициализация Ксавье (Глоро).** Глоро и Бенжио (2010) потребовали сохранить дисперсию и при прямом проходе ($n_{in}\sigma^2 = 1$), и при обратном ($n_{out}\sigma^2 = 1$). Оба условия выполняются сразу только при $n_{in} = n_{out}$, поэтому берут компромисс (формула 8.23 у Гудфеллоу):

$$\sigma^2 = \frac{2}{n_{in} + n_{out}}.$$

Обычно веса берут из равномерного распределения $U(-a, a)$, у которого дисперсия $a^2/3$, поэтому $a = \sqrt{6 / (n_{in} + n_{out})}$. Для слоя $256 \times 256$: $a = \sqrt{6/512} \approx 0{,}108$, $\sigma = 0{,}0625$. Формула выведена для сети без нелинейностей. Функция tanh немного сжимает сигнал, поэтому PyTorch советует умножать границу на **коэффициент усиления** $5/3$ (`nn.init.calculate_gain('tanh')`).

**4. Инициализация Кайминга (Хе).** Хе и соавторы (2015) учли, что ReLU обнуляет отрицательную половину значений: если $z$ симметрично относительно нуля, то средний квадрат выхода $\mathbb{E}[h^2] = \mathrm{Var}(z)/2$. Чтобы дисперсия сохранялась, нужно $n_{in}\,\sigma^2 \cdot \tfrac12 = 1$:

$$\sigma = \sqrt{\frac{2}{n_{in}}}, \qquad \text{для слоя с 256 входами } \sigma \approx 0{,}0884.$$

Если в сети с ReLU взять Ксавье, дисперсия делится пополам на каждом слое, и через 50 слоев отклонение уменьшится в $\sqrt{2}^{\,50} \approx 3 \cdot 10^7$ раз.

*Как PyTorch инициализирует слои по умолчанию.* `nn.Linear` и `nn.Conv2d` берут веса из $U(-1/\sqrt{n_{in}},\ 1/\sqrt{n_{in}})$, это первая эвристика из раздела 8.4. Дисперсия такого распределения $1/(3n_{in})$, и в сети с ReLU отклонение умножается на $\sqrt{1/6} \approx 0{,}41$ за слой. Для неглубоких сетей вроде `SimpleCNN` это не мешает, а для сети из десятков слоев без нормализации мешает сильно.

*Входы и выходы сверточного слоя.* Нейрон свертки видит $C_{in} \cdot k \cdot k$ чисел, поэтому $n_{in} = C_{in} k^2$ (по-английски fan_in), $n_{out} = C_{out} k^2$ (fan_out). Для `Conv2d(8, 16, 3)`: $n_{in} = 72$, $n_{out} = 144$, граница Ксавье $\sqrt{6/216} = 1/6$, отклонение Кайминга $\sqrt{2/72} = 1/6$ (совпадение случайное).

**5. Утесы и взрыв градиента.** У функции потерь глубокой или рекуррентной сети бывают «утесы»: участки, где градиент огромен, и один шаг уносит веса далеко от хорошей области (раздел 8.2.4). Простая защита называется **отсечением градиента по норме**: если общая норма градиента $\lVert g \rVert$ больше порога $c$, градиент уменьшают до длины $c$, сохраняя направление:

$$g \leftarrow g \cdot \frac{c}{\lVert g \rVert}.$$

Норма считается сразу по всем параметрам сети: $\lVert g \rVert = \sqrt{\sum_p \sum_i g_{p,i}^2}$. Пример: $g = (3, 4)$, $\lVert g \rVert = 5$, при $c = 1$ получаем $(0{,}6;\ 0{,}8)$. В PyTorch это `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)`, ее вызывают между `loss.backward()` и `opt.step()`. Особенно часто отсечение нужно рекуррентным сетям (Б1.5).

**6. Пакетная нормализация.** Хорошая инициализация задает удачный масштаб только в начале. Дальше все слои меняются одновременно, и распределение выходов каждого слоя «плывет» под следующими слоями (раздел 8.7.1, в русском переводе книги метод называется «пакетная нормировка»). Иоффе и Сегеди (2015) предложили прямо в сети приводить выход каждого нейрона к среднему 0 и дисперсии 1 по текущему мини-пакету из $m$ объектов:

$$\mu_B = \frac1m \sum_{i=1}^m h_i, \qquad \sigma_B^2 = \frac1m \sum_{i=1}^m (h_i - \mu_B)^2, \qquad \hat h_i = \frac{h_i - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \qquad y_i = \gamma\,\hat h_i + \beta.$$

- $\gamma$ (масштаб) и $\beta$ (сдвиг) обучаются вместе с весами. Они позволяют сети вернуть любое среднее и отклонение, если это полезно: при $\gamma = \sigma_B$ и $\beta = \mu_B$ слой выдаст исходные значения. Начальные значения $\gamma = 1$, $\beta = 0$.
- $\varepsilon = 10^{-5}$ защищает от деления на ноль.
- Через $\mu_B$ и $\sigma_B$ градиент тоже распространяется: нормализация входит в граф вычислений.

*Пример со слайдов.* Выход нейрона на четырех объектах пакета $h = (3, 5, 5, 11)$: $\mu_B = 6$, $\sigma_B^2 = (9 + 1 + 1 + 25)/4 = 9$, $\sigma_B = 3$, $\hat h = (-1;\ -0{,}333;\ -0{,}333;\ 1{,}667)$. При $\gamma = 2$, $\beta = 0{,}5$: $y = (-1{,}5;\ -0{,}167;\ -0{,}167;\ 3{,}833)$.

*Где ставить.* Обычно после линейного или сверточного слоя и перед функцией активации. Смещение предыдущего слоя тогда не нужно: вычитание среднего его уничтожает, а его роль играет $\beta$. Поэтому пишут `nn.Conv2d(..., bias=False)`.

*Свертки.* У сверточного слоя нормализуется каждый канал: свои $\gamma$ и $\beta$ на канал, а среднее и дисперсия считаются по всем объектам пакета и всем позициям картинки, то есть по $N \cdot H \cdot W$ числам. В PyTorch это `nn.BatchNorm2d(C)`; для полносвязного слоя `nn.BatchNorm1d(C)` считает статистику по $N$ объектам.

**7. Два режима работы.** При обучении берется статистика текущего пакета. При применении объект может быть один, и ответ не должен зависеть от соседей по пакету. Поэтому во время обучения слой копит **скользящие средние**:

$$\text{running\_mean} \leftarrow (1 - m)\cdot\text{running\_mean} + m\cdot\mu_B, \qquad \text{running\_var} \leftarrow (1 - m)\cdot\text{running\_var} + m\cdot\sigma_B^2\cdot\frac{n}{n-1},$$

где $m = 0{,}1$ (параметр `momentum` в PyTorch), $n$ это число значений, по которым считалась статистика. Обратите внимание: здесь $m$ задает вес **нового** значения, а у импульса в Б2.1 $\beta$ задавал вес **старого**. Дисперсия в скользящем среднем **несмещенная** (деление на $n - 1$), а для нормализации пакета берется смещенная (деление на $n$). Начальные значения: running_mean = 0, running_var = 1. В примере после одного шага running_mean $= 0{,}9 \cdot 0 + 0{,}1 \cdot 6 = 0{,}6$, несмещенная дисперсия $36 / 3 = 12$, running_var $= 0{,}9 \cdot 1 + 0{,}1 \cdot 12 = 2{,}1$.

При применении $\hat h = (h - \text{running\_mean}) / \sqrt{\text{running\_var} + \varepsilon}$. Режим переключают `model.train()` и `model.eval()`. Забытый `model.eval()` перед проверкой это частая ошибка: ответ на картинку начинает зависеть от других картинок пакета, а скользящие средние портятся тестовыми данными.

**8. Почему нормализация помогает и чем за нее платят.**
- Помогает: можно брать скорость обучения больше, сеть меньше зависит от начальных весов, потеря падает быстрее; шум статистики пакета слегка регуляризует сеть. Точная причина до сих пор обсуждается: авторы объясняли успех борьбой со «сдвигом распределений» между слоями, позже Сантуркар и соавторы (2018) показали, что главное, видимо, в сглаживании поверхности потерь.
- Платят: при маленьких пакетах (2–4 объекта) статистика очень шумная; поведение в обучении и применении разное; нужны лишние вычисления; в рекуррентных сетях и трансформерах пакетная нормализация неудобна.

**9. Нормализация по слою.** Ба, Кирос и Хинтон (2016) предложили те же формулы, но среднее и дисперсию считать по признакам **одного объекта**, а не по объектам пакета. Такая нормализация не зависит от размера пакета, одинаково работает при обучении и применении и не требует скользящих средних. Она стоит в каждом блоке трансформера (Тема 2) и часто в рекуррентных сетях. В PyTorch это `nn.LayerNorm(C)`.

**10. В PyTorch.** Модуль `torch.nn.init` содержит `xavier_uniform_`, `xavier_normal_`, `kaiming_normal_(w, nonlinearity='relu')`, `zeros_`; подчеркивание в конце имени означает, что тензор меняется на месте. Параметры $\gamma$ и $\beta$ лежат в `model.parameters()` и обновляются оптимизатором. Скользящие средние хранятся как **буферы** (`register_buffer`): оптимизатор их не трогает, но они сохраняются в `model.state_dict()` и переносятся вместе с моделью на видеокарту.

In [ ]:
# Нужна, только если ноутбук запускается отдельно от ПР
from pathlib import Path
import copy
import math
import time
import numpy as np
import matplotlib
matplotlib.use('Agg')   # графики сохраняются в .png и показываются через display; для интерактива закомментируйте
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from IPython.display import Image, display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
print('torch', torch.__version__)

# Цвета графиков – как на слайдах
NAVY, BLUE, DEEP, GRAY, LIGHT = '#0F1B2D', '#4C9BFF', '#1F63C9', '#9AA0A8', '#EAF2FF'

def show(t, k=3):
    """Печать тензора списком с округлением – так удобно сверять с «ожидается»."""
    t = torch.as_tensor(t).detach()
    if t.dim() == 0:
        return round(t.item(), k)
    return torch.round(t.double() * 10**k).div(10**k).tolist()

# FashionMNIST для Н6 заранее скачивается в папку data (интернет в аудитории не нужен):
#   datasets.FashionMNIST('data', train=True, download=True); datasets.FashionMNIST('data', train=False, download=True)
DATA_DIR = Path('data')
tf = transforms.ToTensor()
try:
    train_ds = datasets.FashionMNIST(DATA_DIR, train=True, download=False, transform=tf)
    test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=False, transform=tf)
except Exception:
    print('FashionMNIST не найден в папке data – используются случайные картинки FakeData')
    train_ds = datasets.FakeData(size=10000, image_size=(1, 28, 28), num_classes=10, transform=tf)
    test_ds = datasets.FakeData(size=2000, image_size=(1, 28, 28), num_classes=10, transform=tf, random_offset=10000)
small_train = Subset(train_ds, range(10000))    # 10 000 картинок: одна эпоха – 157 итераций
small_test = Subset(test_ds, range(2000))
test_loader = DataLoader(small_test, batch_size=500)

# Сеть и функция из части 4А ПР1
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

def accuracy(model, loader):
    model.eval()
    correct, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(dim=1) == y).sum().item()
            n += len(y)
    return correct / n

# Глубокая полносвязная сеть для Н2–Н4: depth одинаковых слоев width -> width без смещений
class DeepMLP(nn.Module):
    def __init__(self, depth=50, width=256, act='tanh'):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(width, width, bias=False) for _ in range(depth)])
        self.act = {'tanh': torch.tanh, 'relu': torch.relu, 'linear': lambda z: z}[act]

    def forward(self, x, collect=False):
        zs, hs = [], []                 # выходы слоев до и после функции активации
        for layer in self.layers:
            z = layer(x)
            x = self.act(z)
            zs.append(z)
            hs.append(x)
        return (x, zs, hs) if collect else x

torch.manual_seed(RANDOM_STATE)
x_big = torch.randn(1000, 256)          # 1000 объектов, 256 признаков со средним 0 и отклонением 1

In [ ]:
# Учебный пример со слайдов: выход одного нейрона на пакете из четырех объектов
h_example = torch.tensor([3.0, 5.0, 5.0, 11.0])
print(show(h_example.mean()), show(h_example.var(unbiased=False)), show(h_example.var()),
      '  ожидается 6.0 9.0 12.0 (среднее, смещенная и несмещенная дисперсия)')

**Задание Н1. Симметрия.**

- `constant_init_(model, value)` записывает число `value` во **все** параметры модели (веса и смещения) на месте: `p.fill_(value)` внутри `torch.no_grad()`.
- `row_spread(W)` показывает, насколько строки матрицы весов отличаются от первой строки: наибольший модуль разности `W - W[0]` (строка `W[0]` вычитается из каждой строки). Ноль означает, что все нейроны слоя одинаковые.

*Проверка по п. 1 теории.* Сеть $2 \to 2 \to 1$ с весами $0{,}5$: выход $1{,}5$, градиенты строк первого слоя $(0{,}5;\ 1)$ у обоих нейронов, градиент выходного слоя $(1{,}5;\ 1{,}5)$, после шага $\eta = 0{,}1$ строки равны $(0{,}45;\ 0{,}4)$. Затем готовая ячейка учит сеть с восемью скрытыми нейронами решать задачу XOR из Б1.1 при постоянной и при случайной инициализации.

In [ ]:
def constant_init_(model, value):
    with torch.no_grad():
        for p in model.parameters():
            ...                                     # <-- записать value во все элементы p на месте
    return model

def row_spread(W):
    """Наибольшее отличие строки матрицы от первой строки: 0 – все нейроны слоя одинаковые."""
    return ...                                      # <-- модуль разности со строкой W[0], максимум, .item()


tiny = constant_init_(nn.Sequential(nn.Linear(2, 2, bias=False), nn.ReLU(), nn.Linear(2, 1, bias=False)), 0.5)
x1, y1 = torch.tensor([[1.0, 2.0]]), torch.tensor([[1.0]])
out = tiny(x1)
((out - y1) ** 2).mean().backward()
print(show(out), show(tiny[0].weight.grad), show(tiny[2].weight.grad),
      '  ожидается [[1.5]] [[0.5, 1.0], [0.5, 1.0]] [[1.5, 1.5]]')
with torch.no_grad():
    for p in tiny.parameters():
        p -= 0.1 * p.grad
print(show(tiny[0].weight), row_spread(tiny[0].weight), '  ожидается [[0.45, 0.4], [0.45, 0.4]] 0.0')

In [ ]:
# Готовая ячейка: XOR с восемью скрытыми нейронами при постоянной и при случайной инициализации
X_xor = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
y_xor = torch.tensor([0.0, 1.0, 1.0, 0.0])

def train_xor(init_value=None, steps=1000):
    torch.manual_seed(RANDOM_STATE)
    net = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
    if init_value is not None:
        constant_init_(net, init_value)
    opt = torch.optim.Adam(net.parameters(), lr=0.05)
    for _ in range(steps):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(net(X_xor).squeeze(1), y_xor).backward()
        opt.step()
    with torch.no_grad():
        acc = ((net(X_xor).squeeze(1) > 0).float() == y_xor).float().mean().item()
    return row_spread(net[0].weight), acc

for label, value in [('все параметры 0.5', 0.5), ('случайные (PyTorch)', None)]:
    spread, acc = train_xor(value)
    print(f'{label:<22} различие нейронов {spread:.1e}   точность {acc:.2f}')
print('  ожидается: при постоянной инициализации различие 0 (или порядка 1e-7) и точность не выше 0.75; при случайной точность 1.00')

**Вопрос Н1.** а) Почему при постоянной инициализации нейроны остались одинаковыми даже у Adam, у которого для каждого веса свой шаг? б) Почему сеть с восемью одинаковыми нейронами не решает XOR, хотя восьми разных нейронов для этого хватает с запасом? в) Почему смещения можно начинать с нуля, а веса нельзя?

*Ваш ответ:*

**Задание Н2. Сигнал в глубокой сети.**

Готовый класс `DeepMLP(depth, width, act)` из ячейки подготовки – это `depth` одинаковых слоев `nn.Linear(width, width, bias=False)` с функцией активации `act` после каждого (`'tanh'`, `'relu'` или `'linear'`, то есть без функции активации). Вызов `model(x, collect=True)` возвращает выход сети и два списка: выходы слоев до функции активации `zs` и после нее `hs`.

- `init_normal_(model, std)` – веса каждого слоя `model.layers` из нормального распределения со средним 0 и отклонением `std`: `layer.weight.normal_(0.0, std)` внутри `torch.no_grad()`.
- `layer_stds(model, x)` – список стандартных отклонений выходов всех слоев до функции активации: `z.std().item()` для каждого `z` из `zs`.
- `predicted_std(std, width, depth)` – прогноз по формуле из п. 2 теории для сети без функции активации и входа с отклонением 1: $\left(\sqrt{\text{width}}\cdot\text{std}\right)^{\text{depth}}$.

*Проверка по п. 2 теории.* Прогнозы для 20 слоев ширины 256: $\sigma = 0{,}05$ дает $0{,}0115$, $\sigma = 0{,}0625$ дает 1, $\sigma = 0{,}1$ дает $12\,089$. Измерения на 1000 случайных объектах должны быть близки к прогнозам: отклоняться они могут на десятки процентов, потому что веса случайные.

In [ ]:
def init_normal_(model, std):
    with torch.no_grad():
        for layer in model.layers:
            ...                                     # <-- нормальное распределение (0, std) на месте
    return model

def layer_stds(model, x):
    """Стандартное отклонение выхода каждого слоя до функции активации."""
    with torch.no_grad():
        _, zs, _ = model(x, collect=True)
    return ...                                      # <-- список z.std().item() по всем слоям

def predicted_std(std, width, depth):
    """Прогноз по формуле из п. 2 теории: сеть без функции активации, вход с отклонением 1."""
    return ...                                      # <-- (корень из width * std) в степени depth


for std in [0.05, 0.0625, 0.1]:
    torch.manual_seed(RANDOM_STATE)
    net = init_normal_(DeepMLP(depth=20, width=256, act='linear'), std)
    s = layer_stds(net, x_big)
    pred = predicted_std(std, 256, 20)
    print(f'std = {std:<7} слой 1: {s[0]:.3f}   слой 20: {s[-1]:.4g}   прогноз {pred:.4g}   '
          f'измерение/прогноз в пределах от 0.5 до 2: {0.5 < s[-1] / pred < 2}')
print('  ожидается: прогнозы 0.01153, 1, 1.209e+04; на слое 1 отклонения около 0.8, 1.0 и 1.6; во всех строках True')

In [ ]:
# Готовая ячейка: гистограммы активаций tanh-сети из 50 слоев при разных масштабах весов
configs = [('σ = 0.01', 0.01), ('σ = 1', 1.0), ('Ксавье: σ = 1/16', 0.0625), ('Ксавье · 5/3', 0.0625 * 5 / 3)]
show_layers = [1, 10, 30, 50]
fig, axs = plt.subplots(len(configs), len(show_layers), figsize=(13, 8.5), sharex=True)
for row, (label, std) in enumerate(configs):
    torch.manual_seed(RANDOM_STATE)
    net = init_normal_(DeepMLP(depth=50, width=256, act='tanh'), std)
    with torch.no_grad():
        _, _, hs = net(x_big, collect=True)
    last = hs[-1]
    print(f'{label:<18} отклонение на слое 50: {last.std().item():.3g}   '
          f'доля насыщенных (|h| > 0.99): {(last.abs() > 0.99).float().mean().item():.2f}')
    for col, k in enumerate(show_layers):
        ax = axs[row, col]
        ax.hist(hs[k - 1].flatten().numpy(), bins=60, range=(-1, 1), color=BLUE if row >= 2 else GRAY)
        ax.set_yticks([])
        if row == 0:
            ax.set_title(f'слой {k}', fontsize=10)
        if col == 0:
            ax.set_ylabel(label, fontsize=10)
fig.tight_layout(); fig.savefig('tanh_histograms.png', dpi=100); plt.close(fig)
display(Image('tanh_histograms.png'))
print('  ожидается: σ = 0.01 – все значения в нуле (отклонение практически 0); σ = 1 – значения прилипли к ±1,')
print('  насыщено больше половины; Ксавье – колокол постепенно сужается (около 0.1 на слое 50); Ксавье · 5/3 – форма сохраняется (около 0.65)')

**Вопрос Н2.** а) Объясните результат для $\sigma = 0{,}1$ в линейной сети через множитель $\sqrt{n}\,\sigma$. Каким должно быть $\sigma$ для слоя с 1024 входами, чтобы сигнал сохранялся? б) Почему в сети с tanh при $\sigma = 1$ сигнал не взрывается, как в линейной, но сеть все равно плохая? в) Почему Ксавье для tanh чуть-чуть «не дотягивает» и сигнал медленно затухает?

*Ваш ответ:*

**Задание Н3. Свои инициализации Ксавье и Кайминга (главное задание).**

- `fans(w)` возвращает пару $(n_{in}, n_{out})$ для тензора весов (п. 4 теории). У линейного слоя форма весов `(out, in)`, у сверточного `(out, in, k, k)`. Число элементов одного ядра `w[0][0].numel()` равно 1 у линейного слоя и $k^2$ у свертки; $n_{in}$ = `w.shape[1]` · размер ядра, $n_{out}$ = `w.shape[0]` · размер ядра.
- `xavier_uniform_(w, gain=1.0)` – равномерно на $[-a, a]$, $a = \text{gain}\cdot\sqrt{6/(n_{in} + n_{out})}$ (п. 3): `w.uniform_(-a, a)` внутри `torch.no_grad()`.
- `kaiming_normal_(w)` – нормально со средним 0 и отклонением $\sqrt{2/n_{in}}$ (п. 4): `w.normal_(0.0, std)`.

Обе функции меняют тензор на месте и возвращают его.

*Проверка.* `Conv2d(8, 16, 3)`: $(72, 144)$; `Linear(256, 256)`: $(256, 256)$. Для матрицы $1000 \times 1000$ отклонение Ксавье $\sqrt{2/2000} \approx 0{,}0316$, Кайминга $\sqrt{2/1000} \approx 0{,}0447$; ваши функции и функции PyTorch должны дать одинаковые числа с точностью до третьего знака. Затем готовая ячейка сравнивает инициализации в сети с ReLU из 50 слоев.

In [ ]:
def fans(w):
    receptive = ...                                 # <-- число элементов одного ядра: 1 у линейного слоя, k*k у свертки
    fan_in = ...                                    # <-- входные каналы (признаки) * размер ядра
    fan_out = ...                                   # <-- выходные каналы (нейроны) * размер ядра
    return fan_in, fan_out

def xavier_uniform_(w, gain=1.0):
    fan_in, fan_out = fans(w)
    a = ...                                         # <-- граница равномерного распределения
    with torch.no_grad():
        ...                                         # <-- равномерно на [-a, a], на месте
    return w

def kaiming_normal_(w):
    fan_in, _ = fans(w)
    std = ...                                       # <-- отклонение для ReLU
    with torch.no_grad():
        ...                                         # <-- нормально (0, std), на месте
    return w


conv_w = nn.Conv2d(8, 16, 3).weight
lin_w = nn.Linear(256, 256).weight
print(fans(conv_w), fans(lin_w), '  ожидается (72, 144) (256, 256)')
torch.manual_seed(RANDOM_STATE)
print(round(xavier_uniform_(torch.empty(1000, 1000)).std().item(), 3),
      round(nn.init.xavier_uniform_(torch.empty(1000, 1000)).std().item(), 3), '  ожидается 0.032 0.032')
print(round(kaiming_normal_(torch.empty(1000, 1000)).std().item(), 3),
      round(nn.init.kaiming_normal_(torch.empty(1000, 1000), nonlinearity='relu').std().item(), 3), '  ожидается 0.045 0.045')
print(round(xavier_uniform_(conv_w).abs().max().item(), 3), '  ожидается чуть меньше 0.167 (граница 1/6)')

In [ ]:
# Готовая ячейка: сеть с ReLU из 50 слоев при трех инициализациях
results = {}
for label, init_fn in [('PyTorch по умолчанию', None), ('Ксавье', xavier_uniform_), ('Кайминг', kaiming_normal_)]:
    torch.manual_seed(RANDOM_STATE)
    net = DeepMLP(depth=50, width=256, act='relu')
    if init_fn is not None:
        for layer in net.layers:
            init_fn(layer.weight)
    results[label] = layer_stds(net, x_big)
    print(f'{label:<22} слой 1: {results[label][0]:.3g}   слой 10: {results[label][9]:.3g}   слой 50: {results[label][-1]:.3g}')
print('  ожидается на слое 50: по умолчанию порядка 1e-20, Ксавье порядка 1e-8, Кайминг около 1–2')

fig, ax = plt.subplots(figsize=(7, 3.4))
for (label, s), c in zip(results.items(), [GRAY, DEEP, BLUE]):
    ax.semilogy(range(1, 51), s, color=c, lw=2.2, label=label)
ax.set_xlabel('номер слоя'); ax.set_ylabel('отклонение выхода слоя'); ax.legend(fontsize=9)
ax.set_title('ReLU, 50 слоев по 256 нейронов')
fig.tight_layout(); fig.savefig('relu_init.png', dpi=110); plt.close(fig)
display(Image('relu_init.png'))

**Вопрос Н3.** а) Объясните наклон каждой линии на графике через множитель за слой: $\sqrt{1/6}$, $\sqrt{1/2}$ и 1. б) Почему у Кайминга на первом слое отклонение около $1{,}4$, а не 1? в) Какую инициализацию вы выберете для `SimpleCNN` с ReLU и для рекуррентной сети с tanh из Б1.5?

*Ваш ответ:*

**Задание Н4. Градиенты по слоям и отсечение градиента.**

- `grad_norms(model)` – список норм градиентов весов всех слоев `model.layers` после `backward()`: `layer.weight.grad.norm().item()`.
- `my_clip_grad_norm_(params, max_norm)` – отсечение по общей норме (п. 5 теории). Общая норма – корень из суммы квадратов **всех** элементов градиентов всех параметров. Коэффициент `coef = max_norm / (total + 1e-6)` (так же, как в PyTorch); если он меньше 1, каждый градиент умножается на него на месте: `p.grad.mul_(coef)`. Функция возвращает общую норму до отсечения.

*Проверка.* Два параметра с градиентами $(3, 0)$ и $(0, 4)$: общая норма 5, при пороге $2{,}5$ градиенты становятся $(1{,}5;\ 0)$ и $(0;\ 2)$. Затем отсечение сверяется с `torch.nn.utils.clip_grad_norm_` на сети `SimpleCNN`, и готовая ячейка показывает градиенты по слоям в tanh-сети из 50 слоев.

In [ ]:
def grad_norms(model):
    return ...                                      # <-- норма градиента весов каждого слоя

def my_clip_grad_norm_(params, max_norm):
    params = [p for p in params if p.grad is not None]
    total = ...                                     # <-- корень из суммы квадратов всех градиентов
    coef = max_norm / (total + 1e-6)
    if coef < 1:
        for p in params:
            ...                                     # <-- уменьшить градиент на месте
    return total


a = torch.zeros(2, requires_grad=True)
b = torch.zeros(2, requires_grad=True)
a.grad, b.grad = torch.tensor([3.0, 0.0]), torch.tensor([0.0, 4.0])
print(show(my_clip_grad_norm_([a, b], 2.5)), show(a.grad), show(b.grad), '  ожидается 5.0 [1.5, 0.0] [0.0, 2.0]')

def cnn_with_grads():
    torch.manual_seed(RANDOM_STATE)
    net = SimpleCNN()
    x, y = torch.randn(16, 1, 28, 28), torch.randint(0, 10, (16,))
    F.cross_entropy(net(x), y).backward()
    return net

n1, n2 = cnn_with_grads(), cnn_with_grads()
t1 = my_clip_grad_norm_(n1.parameters(), 0.1)
t2 = torch.nn.utils.clip_grad_norm_(n2.parameters(), 0.1)
diff = max((p.grad - q.grad).abs().max().item() for p, q in zip(n1.parameters(), n2.parameters()))
print(f'общие нормы: {t1.item():.4f} и {t2.item():.4f}, расхождение градиентов {diff:.1e}', '  ожидается: нормы равны, расхождение < 1e-06')

In [ ]:
# Готовая ячейка: нормы градиентов по слоям в tanh-сети из 50 слоев
grads = {}
for label, std in [('σ = 0.01', 0.01), ('σ = 1', 1.0), ('Ксавье', 0.0625)]:
    torch.manual_seed(RANDOM_STATE)
    net = init_normal_(DeepMLP(depth=50, width=256, act='tanh'), std)
    net(x_big[:256]).mean().backward()
    grads[label] = grad_norms(net)
    print(f'{label:<10} слой 1: {grads[label][0]:.3g}   слой 25: {grads[label][24]:.3g}   слой 50: {grads[label][-1]:.3g}')
print(max(grads['σ = 0.01']) < 1e-20, grads['σ = 1'][0] / grads['σ = 1'][-1] > 1e6,
      0.1 < grads['Ксавье'][0] / grads['Ксавье'][-1] < 10, '  ожидается True True True')

fig, ax = plt.subplots(figsize=(7, 3.4))
for (label, g), c in zip(grads.items(), [GRAY, NAVY, BLUE]):
    ax.semilogy(range(1, 51), np.maximum(g, 1e-45), color=c, lw=2.2, label=label)
ax.set_xlabel('номер слоя'); ax.set_ylabel('норма градиента весов'); ax.legend(fontsize=9)
ax.set_title('tanh, 50 слоев: градиенты сразу после инициализации')
fig.tight_layout(); fig.savefig('tanh_grads.png', dpi=110); plt.close(fig)
display(Image('tanh_grads.png'))

**Вопрос Н4.** а) Почему при $\sigma = 1$ градиенты первых слоев в миллиарды миллиардов раз больше, чем у последнего? Что случится после первого же шага SGD? б) Почему при $\sigma = 0{,}01$ градиенты практически нулевые у всех слоев, даже у последнего? в) Почему отсечение градиента сохраняет направление шага и в чем его отличие от уменьшения скорости обучения?

*Ваш ответ:*

**Задание Н5. Свой слой пакетной нормализации (главное задание).**

Допишите класс `MyBatchNorm(num_features, eps=1e-5, momentum=0.1)` по пп. 6–7 теории. Он должен работать и для выхода полносвязного слоя формы `(N, C)`, и для выхода свертки формы `(N, C, H, W)`. Уже готово:
- параметры `weight` ($\gamma$, начальное значение 1) и `bias` ($\beta$, начальное значение 0) – `nn.Parameter`, их будет обучать оптимизатор;
- буферы `running_mean` (нули) и `running_var` (единицы) – `register_buffer`: они сохраняются вместе с моделью, но не обучаются;
- `dims` – по каким осям считать статистику: `[0]` для `(N, C)` и `[0, 2, 3]` для `(N, C, H, W)`;
- `shape` – форма для умножения на `x`: `(1, C)` или `(1, C, 1, 1)`; например, `mean.view(shape)`.

Нужно написать:
- в режиме обучения (`self.training` равно `True`): среднее `x.mean(dim=dims)` и **смещенную** дисперсию `x.var(dim=dims, unbiased=False)`; обновление скользящих средних на месте внутри `torch.no_grad()` – `self.running_mean.mul_(1 - m).add_(m * mean)` и то же для `running_var`, но с **несмещенной** дисперсией `var * n / (n - 1)`;
- в режиме применения: взять `running_mean` и `running_var`;
- нормализацию $\hat x = (x - \mu)/\sqrt{\sigma^2 + \varepsilon}$ и выход $\gamma \hat x + \beta$.

Режим переключают унаследованные от `nn.Module` методы `.train()` и `.eval()`, они меняют `self.training`.

*Проверка по пп. 6–7 теории.* Пакет $(3, 5, 5, 11)$ как тензор формы $(4, 1)$, $\gamma = 2$, $\beta = 0{,}5$: выход $(-1{,}5;\ -0{,}167;\ -0{,}167;\ 3{,}833)$, после этого шага running_mean $= 0{,}6$, running_var $= 2{,}1$. В режиме применения тот же пакет дает для первого объекта $\big((3 - 0{,}6)/\sqrt{2{,}1}\big)\cdot 2 + 0{,}5 \approx 3{,}812$. Затем ваш слой сверяется с `nn.BatchNorm1d` и `nn.BatchNorm2d`: выходы, градиенты и скользящие средние должны совпасть с точностью до округлений в `float32`.

In [ ]:
class MyBatchNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps, self.momentum = eps, momentum
        self.weight = nn.Parameter(torch.ones(num_features))     # gamma
        self.bias = nn.Parameter(torch.zeros(num_features))      # beta
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x):
        dims = [0] if x.dim() == 2 else [0, 2, 3]            # оси, по которым считается статистика канала
        shape = [1, -1] + [1] * (x.dim() - 2)                 # (1, C) или (1, C, 1, 1)
        m = self.momentum
        if self.training:
            mean = ...                              # <-- среднее по пакету (и по позициям у свертки)
            var = ...                               # <-- смещенная дисперсия: деление на n
            n = x.numel() // x.shape[1]                       # сколько значений вошло в статистику канала
            with torch.no_grad():
                ...                                 # <-- скользящее среднее
                ...                                 # <-- с несмещенной дисперсией
        else:
            mean, var = ...                         # <-- режим применения
        x_hat = ...                                 # <-- нормализация
        return ...                                  # <-- масштаб и сдвиг


bn = MyBatchNorm(1)
with torch.no_grad():
    bn.weight.fill_(2.0)
    bn.bias.fill_(0.5)
h4 = h_example.view(4, 1)
print(show(bn(h4).view(-1)), '  ожидается [-1.5, -0.167, -0.167, 3.833]')
print(show(bn.running_mean), show(bn.running_var), '  ожидается [0.6] [2.1]')
bn.eval()
print(show(bn(h4).view(-1)), '  ожидается [3.812, 6.573, 6.573, 14.853]')

In [ ]:
# Готовая ячейка: сверка с nn.BatchNorm1d и nn.BatchNorm2d
def compare_bn(ref_cls, input_shape, steps=5):
    torch.manual_seed(RANDOM_STATE)
    C = input_shape[1]
    mine, ref = MyBatchNorm(C), ref_cls(C)
    with torch.no_grad():
        ref.weight.copy_(mine.weight.uniform_(0.5, 2.0))
        ref.bias.copy_(mine.bias.uniform_(-1.0, 1.0))
    out_diff = grad_diff = 0.0
    for _ in range(steps):                                       # режим обучения: несколько пакетов
        x = 3 + 2 * torch.randn(*input_shape)
        xa, xb = x.clone().requires_grad_(), x.clone().requires_grad_()
        ya, yb = mine(xa), ref(xb)
        w = torch.randn_like(ya)                                 # случайная «потеря» (w * y).sum()
        (w * ya).sum().backward()
        (w * yb).sum().backward()
        out_diff = max(out_diff, (ya - yb).abs().max().item())
        grad_diff = max(grad_diff, (xa.grad - xb.grad).abs().max().item(),
                        (mine.weight.grad - ref.weight.grad).abs().max().item())
        mine.zero_grad(); ref.zero_grad()
    run_diff = max((mine.running_mean - ref.running_mean).abs().max().item(),
                   (mine.running_var - ref.running_var).abs().max().item())
    mine.eval(); ref.eval()
    x = 3 + 2 * torch.randn(*input_shape)
    with torch.no_grad():
        eval_diff = (mine(x) - ref(x)).abs().max().item()
    return out_diff, grad_diff, run_diff, eval_diff

for ref_cls, shape in [(nn.BatchNorm1d, (32, 6)), (nn.BatchNorm2d, (8, 4, 5, 5))]:
    d = compare_bn(ref_cls, shape)
    print(f'{ref_cls.__name__}: выход {d[0]:.1e}, градиенты {d[1]:.1e}, скользящие средние {d[2]:.1e}, применение {d[3]:.1e}')
print('  ожидается: все расхождения < 1e-04 (обычно порядка 1e-06 и меньше)')

**Вопрос Н5.** а) Почему для нормализации пакета берется смещенная дисперсия, а в скользящее среднее кладется несмещенная? б) Чему будет равен выход слоя в режиме обучения, если в пакете один объект формы `(1, C)`? Что сделает PyTorch в этом случае? в) После одного шага скользящие средние $(0{,}6;\ 2{,}1)$ далеки от статистики пакета $(6;\ 9)$. Сколько шагов на одном и том же пакете нужно, чтобы running_mean отличалось от 6 меньше чем на 1 %?

*Ваш ответ:*

**Задание Н6. Пакетная нормализация в настоящей сети.**

Допишите класс `SimpleCNN_BN` – сеть `SimpleCNN` из части 4А, в которой после каждой свертки стоит `nn.BatchNorm2d` с числом каналов этой свертки, а у сверток нет смещения (`bias=False`, п. 6 теории). Порядок в каждом блоке: свертка, нормализация, ReLU, пулинг.

*Проверка.* Число параметров: у `SimpleCNN` – 9098, у `SimpleCNN_BN` – 9122 (смещения сверток $8 + 16$ ушли, пришли $\gamma$ и $\beta$: $2 \cdot (8 + 16)$). Готовые ячейки:
1. обучают обе сети одну эпоху на 10 000 картинках обычным SGD с тремя скоростями обучения $0{,}01$, $0{,}1$ и $1$ (шесть запусков, около полутора минут);
2. показывают, что в режиме обучения ответ сети на картинку зависит от соседей по пакету, а в режиме применения нет.

In [ ]:
class SimpleCNN_BN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            ...,                                    # <-- блок 1
            ...,                                    # <-- блок 2
        )
        self.classifier = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


count = lambda m: sum(p.numel() for p in m.parameters())
print(count(SimpleCNN()), count(SimpleCNN_BN()), '  ожидается 9098 9122')
print([k for k in SimpleCNN_BN().state_dict() if k.startswith('features.1')])
print("  ожидается ['features.1.weight', 'features.1.bias', 'features.1.running_mean', 'features.1.running_var', 'features.1.num_batches_tracked']")

In [ ]:
# Готовая ячейка: одна эпоха на 10 000 картинок, SGD без импульса, три скорости обучения
def run(model_cls, lr):
    torch.manual_seed(RANDOM_STATE)
    model = model_cls()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loader = DataLoader(small_train, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(RANDOM_STATE))
    model.train()
    hist = []
    for x, y in loader:
        opt.zero_grad()
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        opt.step()
        hist.append(loss.item())
    return model, hist, accuracy(model, test_loader)

res = {}
start = time.perf_counter()
for name, cls in [('без нормализации', SimpleCNN), ('с нормализацией', SimpleCNN_BN)]:
    for lr in [0.01, 0.1, 1.0]:
        res[(name, lr)] = run(cls, lr)
        _, h, acc = res[(name, lr)]
        print(f'{name:<17} lr = {lr:<5} потеря в конце {np.mean(h[-20:]):.3f}   точность на тесте {acc:.3f}')
print(f'время: {time.perf_counter() - start:.0f} с')
end_loss = {k: np.mean(v[1][-20:]) for k, v in res.items()}
print(end_loss[('с нормализацией', 0.01)] < end_loss[('без нормализации', 0.01)],
      '  ожидается True (при малом шаге сеть с нормализацией учится заметно быстрее)')
print('  ожидается также: без нормализации lr = 1 обычно разваливается (nan или потеря около 2.3, как в О6),')
print('  с нормализацией сеть при lr = 1 продолжает учиться, хотя и неровно')

fig, axs = plt.subplots(1, 3, figsize=(14, 3.4), sharey=True)
for ax, lr in zip(axs, [0.01, 0.1, 1.0]):
    for name, c in [('без нормализации', GRAY), ('с нормализацией', BLUE)]:
        h = np.nan_to_num(np.array(res[(name, lr)][1]), nan=3.0)
        ax.plot(np.convolve(h, np.ones(10) / 10, mode='valid'), color=c, lw=2, label=name)
    ax.set_title(f'lr = {lr}'); ax.set_xlabel('итерация'); ax.set_ylim(0, 3)
axs[0].set_ylabel('потеря'); axs[0].legend(fontsize=9)
fig.tight_layout(); fig.savefig('bn_compare.png', dpi=110); plt.close(fig)
display(Image('bn_compare.png'))

In [ ]:
# Готовая ячейка: режим обучения и режим применения
model_bn = copy.deepcopy(res[('с нормализацией', 0.1)][0])   # копия, чтобы не испортить скользящие средние
x16 = next(iter(test_loader))[0][:16]
for mode in ['train', 'eval']:
    getattr(model_bn, mode)()
    with torch.no_grad():
        alone = model_bn(x16[:4])[0]     # первая картинка в пакете из 4
        crowd = model_bn(x16)[0]         # та же картинка в пакете из 16
    print(f'{mode}: наибольшая разница ответов на одну и ту же картинку {(alone - crowd).abs().max().item():.4f}')
print('  ожидается: в режиме train разница заметно больше 0, в режиме eval – 0.0000')

**Вопрос Н6.** а) Почему с нормализацией сеть при малой скорости обучения учится быстрее, хотя веса и оптимизатор те же? б) Почему при $\eta = 1$ сеть с нормализацией обучается неровно? Какой слой в ней не нормализован? в) Функция `accuracy` из части 4А сама вызывает `model.eval()`. Что было бы, если бы она этого не делала, а тестовые картинки шли пакетами, отсортированными по классам?

*Ваш ответ:*

### Задание повышенной сложности (к части 2Н)

1. **Обратный проход пакетной нормализации вручную.** Выведите формулу градиента по входу:
$$\frac{\partial L}{\partial x_i} = \frac{\gamma}{m\sqrt{\sigma_B^2 + \varepsilon}}\left(m\,g_i - \sum_j g_j - \hat x_i \sum_j g_j \hat x_j\right), \qquad g_i = \frac{\partial L}{\partial y_i},$$
напишите класс-наследник `torch.autograd.Function` с собственными `forward` и `backward` и сверьте градиенты с `MyBatchNorm` и с `torch.autograd.gradcheck` (в `float64`). Объясните, почему сумма градиентов по входу равна нулю.
2. **Нормализация по слою.** Напишите `MyLayerNorm(C)` по п. 9 теории (статистика по последней оси, смещенная дисперсия, без скользящих средних) и сверьте с `nn.LayerNorm`. Покажите, что ответ `MyLayerNorm` на объект не зависит от остальных объектов пакета и одинаков в режимах `train` и `eval`.
3. **Маленькие пакеты.** Обучите `SimpleCNN_BN` одну эпоху с пакетами 2, 8 и 64 (скорость обучения пропорциональна размеру пакета). Сравните точность в режимах `train` и `eval` и объясните, почему при пакете 2 они сильно расходятся. Замените `nn.BatchNorm2d` на `nn.GroupNorm(1, C)` и повторите.
4. **Автоматический подбор масштаба (LSUV).** Реализуйте процедуру из раздела 8.4 Гудфеллоу (Mishkin, Matas, 2015): начните с любых случайных весов и, слой за слоем от входа к выходу, делите веса слоя на отклонение его выхода на одном пакете, пока отклонение не станет близким к 1. Проверьте на `DeepMLP` с tanh и с ReLU из 50 слоев, что сигнал после этого сохраняется на всей глубине.

In [ ]:
# Место для задания повышенной сложности